# Mnemonics — bge-reranker-v2-m3 Eval

**Kaggle ayarları:**
- Accelerator → GPU T4 x2
- Internet → ON
- Persistence → Variables and Files

Run All bas, ~60-90dk bekle.

In [ ]:
# 1) Repo kur
!rm -rf /kaggle/working/mnemonics
!git clone https://github.com/nakata-app/mnemonics.git /kaggle/working/mnemonics
%cd /kaggle/working/mnemonics
!git log --oneline -3

In [ ]:
# 2) Bağımlılıklar
!pip install -q -e . sentence-transformers numpy adaptmem 2>&1 | tail -5
print('Install OK')

In [ ]:
# 3) Dataset
import os

# Önce Kaggle dataset'ten bak, yoksa HF'den indir
KAGGLE_DS = '/kaggle/input/datasets/atakanakbaba/mnemonics-lme/longmemeval_s_cleaned.json'
HF_DATA   = '/kaggle/working/longmemeval_s.json'

if os.path.exists(KAGGLE_DS):
    DATA = KAGGLE_DS
    print(f'Kaggle dataset bulundu: {DATA}')
else:
    print('HuggingFace\'den indiriliyor (~265 MB)...')
    !wget -q -O {HF_DATA} \
        'https://huggingface.co/datasets/xiaowu0162/LongMemEval/resolve/main/longmemeval_s.json'
    DATA = HF_DATA

print(f'Size: {os.path.getsize(DATA)/1e6:.1f} MB')

In [ ]:
# 4) DATA path'i eval script'e yaz
import re, pathlib
p = pathlib.Path('/kaggle/working/mnemonics/benchmarks/longmemeval_eval.py')
src = p.read_text()
src = re.sub(r'DATA = _resolve_path.*?\)', f'DATA = Path("{DATA}")', src, flags=re.DOTALL)
p.write_text(src)
!grep -n 'DATA = Path' {p}

In [ ]:
# 5) Smoke test (5 soru, hızlı kontrol)
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=BAAI/bge-reranker-v2-m3 \
    python benchmarks/longmemeval_eval.py \
    --n 5 --mode rerank \
    --augment-preferences --candidate-k 50 \
    --out /tmp/smoke_v2m3.json && echo '=== SMOKE OK ==='

In [ ]:
# 6) 100q ablation — bge-reranker-v2-m3
import os
os.makedirs('/kaggle/working/results', exist_ok=True)

print('=== 100q — BAAI/bge-reranker-v2-m3 ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=BAAI/bge-reranker-v2-m3 \
    python benchmarks/longmemeval_eval.py \
    --n 100 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme100_v2m3.json \
    --per-q-out /kaggle/working/results/lme100_v2m3_perq.json

In [ ]:
# 7) 100q sonuç
import json
r = json.load(open('/kaggle/working/results/lme100_v2m3.json'))['mnemonics_rerank']
print(f'bge-v2-m3  100q  R@1={r["R@1"]:.3f}  R@5={r["R@5"]:.3f}  R@10={r["R@10"]:.3f}')
print(f'MiniLM baseline 100q: R@1=0.880 (referans)')
print(f'Hedef 500q: R@1>0.920 (MemPalace)')
if r['R@1'] >= 0.880:
    print('✅ MiniLM\'den iyi veya eşit → 500q koş')
else:
    print('⚠️  MiniLM\'den düşük, 500q atla')

In [ ]:
# 8) 500q full eval (sadece v2-m3 >= MiniLM ise)
print('=== 500q — BAAI/bge-reranker-v2-m3 ===')
!cd /kaggle/working/mnemonics && \
    MNEMONICS_RERANK_MODEL=BAAI/bge-reranker-v2-m3 \
    python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /kaggle/working/results/lme500_v2m3.json \
    --per-q-out /kaggle/working/results/lme500_v2m3_perq.json

In [ ]:
# 9) Final özet
import json
r = json.load(open('/kaggle/working/results/lme500_v2m3.json'))['mnemonics_rerank']
print('=== FINAL 500q — bge-reranker-v2-m3 ===')
print(f'R@1={r["R@1"]:.3f}  R@5={r["R@5"]:.3f}  R@10={r["R@10"]:.3f}  runtime={r["runtime_s"]:.0f}s')
print()
print('Karşılaştırma:')
print(f'  Baseline (MiniLM 500q): R@1=0.846')
print(f'  MiniLM+augment 100q:    R@1=0.880')
print(f'  MemPalace hedef:        R@1=0.920')
print()
print('=== BY TYPE (R@1) ===')
for qt in sorted(r['by_type']):
    b = r['by_type'][qt]
    print(f'  {qt:28} n={b["n"]:3}  R@1={b["R@1"]:.3f}')

In [ ]:
# 10) Dosyaları listele
!ls -lh /kaggle/working/results/